### data inggestion to vector store pipeline

In [14]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [15]:
### Read all the pdf's inside the directory

def process_all_pdfs(data_dir: str) -> list:
    """Recursively find every PDF under data_dir, load it, and tag each doc with source metadata."""
    data_path = Path(data_dir)
    pdf_files = sorted(data_path.rglob("*.pdf"))

    all_documents = []
    print(f"Found {len(pdf_files)} PDF file(s) in {data_dir}\n")

    for pdf_file in pdf_files:
        try:
            print(f"Processing {pdf_file.name}...")
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

    # Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF file(s) in ../data

Processing AIEngineer.pdf...
  ✓ Loaded 529 pages
Processing pythonbook.pdf...
  ✓ Loaded 61 pages

Total documents loaded: 590


In [16]:
### text splits into chunks

def chunk_documents(documents: list, chunk_size: int = 1000, chunk_overlap: int = 200) -> list:
    """Split loaded documents into smaller overlapping chunks for embedding/retrieval."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )

    chunks = text_splitter.split_documents(documents)

    # Tag each chunk with its position for traceability back to the source
    for i, chunk in enumerate(chunks):
        chunk.metadata['chunk_id'] = i

    print(f"Split {len(documents)} documents into {len(chunks)} chunks")
    return chunks

In [17]:
chunked_documents = chunk_documents(all_pdf_documents)
chunked_documents[0]

Split 590 documents into 561 chunks


Document(metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': '../data/pdf/AIEngineer.pdf', 'file_path': '../data/pdf/AIEngineer.pdf', 'total_pages': 529, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-06-17T05:56:46+00:00', 'trapped': '', 'modDate': 'D:20260617055646Z', 'creationDate': '', 'page': 3, 'source_file': 'AIEngineer.pdf', 'file_type': 'pdf', 'chunk_id': 0}, page_content='AI Engineer\nSalary in India')

### EMBEDDINGS AND VECTOR STORES

In [18]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [19]:
class EmbeddingManager:
    '''handles embedding generation, storage, and retrieval for documents using SentenceTransformer and ChromaDB.'''


    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        '''initialize the embedding manager with a specified model and set up ChromaDB collection.
        Args:
            model_name (str): The name of the SentenceTransformer model to use for embeddings.
        '''

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        '''Load the SentenceTransformer model.'''
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded model: {self.model_name}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        '''Generate embeddings for a list of texts.
        Args:
            texts (List[str]): A list of strings to generate embeddings for.
        Returns:
            np.ndarray: An array of embeddings.
        '''
        if not self.model:
            raise ValueError("Model not loaded. Call _load_model() first.") 
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        print(f"Generated embeddings of shape: {embeddings.shape}")
        return embeddings

        # initialize embedding manager 
        self.embedding_manager = EmbeddingManager()
        embedding_manager

### chroma db and client

In [21]:
class VectorStore:

    def __init__(
        self,
        persist_dir: str = "../data/vectorstore",
        collection_name: str = "rag_collection",
        embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2",
    ):
        self.persist_dir = persist_dir
        self.collection_name = collection_name
        self.embeddings = HuggingFaceEmbeddings(model_name=embedding_model)
        self.store = None

    def build(self, chunks: list) -> "VectorStore":
        """Embed chunks and create a fresh, persisted Chroma collection from them."""
        self.store = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            collection_name=self.collection_name,
            persist_directory=self.persist_dir,
        )
        print(f"Vector store built with {len(chunks)} chunks -> {self.persist_dir}")
        return self

    def load(self) -> "VectorStore":
        """Load an existing persisted Chroma collection instead of rebuilding it."""
        self.store = Chroma(
            collection_name=self.collection_name,
            embedding_function=self.embeddings,
            persist_directory=self.persist_dir,
        )
        print(f"Vector store loaded from {self.persist_dir}")
        return self

    def add_documents(self, chunks: list) -> None:
        """Embed and add more chunks to an existing collection."""
        if self.store is None:
            raise ValueError("Vector store not initialized. Call build() or load() first.")
        self.store.add_documents(chunks)
        print(f"Added {len(chunks)} chunks to vector store")

    def similarity_search(self, query: str, k: int = 4) -> list:
        if self.store is None:
            raise ValueError("Vector store not initialized. Call build() or load() first.")
        return self.store.similarity_search(query, k=k)

    def as_retriever(self, k: int = 4):
        if self.store is None:
            raise ValueError("Vector store not initialized. Call build() or load() first.")
        return self.store.as_retriever(search_kwargs={"k": k})## vector store



In [22]:
vector_store = VectorStore().build(chunked_documents)

results = vector_store.similarity_search("what is python?", k=3)
for r in results:
    print(r.metadata.get("source_file"), "-", r.page_content[:120])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6421.51it/s]


Vector store built with 561 chunks -> ../data/vectorstore
pythonbook.pdf - 6 
 
PYTHON PROGRAMMING HANDBOOK 
WHAT IS PROGRAMMING?  
Just like we use Hindi or English to communicate with each othe
pythonbook.pdf - Python: 
• 
Fundamental Concepts: Start with the basics, such as installing Python and writing your 
first program. 
• 

pythonbook.pdf - 1 
 
PREFACE 
Welcome to the “Ultimate Python Programming Handbook," your comprehensive guide to 
mastering Python progr
